# Notebook 01 — Generate Synthetic Candlestick Dataset

**Goal:** Build the labelled training set for Stage 1 of the pipeline.

We mathematically construct 7 price-pattern classes (6 classic patterns + *No Pattern*),
render them as 224×224 candlestick chart images, and save them to disk.

**Course connections:**
- Lecture 7 — data augmentation rationale (why we add noise during generation)
- Lecture 1 — supervised learning pipeline: train / val split


In [ ]:
import sys, os
# ── Add project root to path (works both locally and on Colab) ───────────────
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Colab: install dependencies
# !pip install -q torch torchvision scipy Pillow tqdm yfinance scikit-learn

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from config import (
    DATA_DIR, N_PER_CLASS, N_CANDLES, IMG_SIZE, VAL_SPLIT, DATA_SEED, CLASS_NAMES
)
from src.data_generation.patterns import generate_pattern_sample, CLASS_NAMES
from src.data_generation.chart_renderer import render_ohlc_to_pil, generate_dataset

np.random.seed(DATA_SEED)
print('Setup complete.')

## 1. Preview — one synthetic sample per class

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for class_id, class_name in enumerate(CLASS_NAMES):
    ohlc = generate_pattern_sample(class_id, n_points=N_CANDLES)
    img  = render_ohlc_to_pil(ohlc, img_size=IMG_SIZE)
    axes[class_id].imshow(img)
    axes[class_id].set_title(class_name.replace('_', '\n'), fontsize=9)
    axes[class_id].axis('off')

axes[-1].axis('off')   # hide the 8th panel
plt.suptitle('One synthetic sample per pattern class', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'figures', 'sample_patterns.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 2. Show price-series diversity (multiple noise levels)

In [ ]:
from src.data_generation.patterns import generate_head_and_shoulders

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
noise_levels = [0.005, 0.012, 0.020, 0.030]

for ax, nl in zip(axes, noise_levels):
    ohlc = generate_pattern_sample(0, n_points=N_CANDLES, noise_level=nl)
    img  = render_ohlc_to_pil(ohlc, img_size=IMG_SIZE)
    ax.imshow(img)
    ax.set_title(f'noise={nl}', fontsize=9)
    ax.axis('off')

plt.suptitle('Head & Shoulders — varying noise levels', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Generate full dataset

This cell writes `N_PER_CLASS` images per class to `data/synthetic/{train,val}/<class>/`.

> **Runtime:** ~10 min for 1 500 images per class (10 500 total) on CPU.  
> On Colab T4 with a GPU the I/O is the bottleneck; expect similar times.

In [ ]:
counts = generate_dataset(
    output_dir  = DATA_DIR,
    n_per_class = N_PER_CLASS,
    n_points    = N_CANDLES,
    img_size    = IMG_SIZE,
    val_split   = VAL_SPLIT,
    seed        = DATA_SEED,
)
print('Train counts:', counts['train'])
print('Val counts:  ', counts['val'])

## 4. Verify class balance

In [ ]:
import pandas as pd
from config import SHORT_NAMES

train_counts = [counts['train'].get(cn, 0) for cn in CLASS_NAMES]
val_counts   = [counts['val'].get(cn, 0)   for cn in CLASS_NAMES]

df = pd.DataFrame({'Train': train_counts, 'Val': val_counts},
                  index=SHORT_NAMES)
df.plot(kind='bar', figsize=(10, 4), color=['steelblue', 'orange'], edgecolor='black')
plt.title('Samples per class (train vs val)')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'figures', 'class_balance.png'),
            dpi=150, bbox_inches='tight')
plt.show()